# ISOM 835 · Session 13 — Ship It: Deployment, Monitoring & Your Prediction Story
**Suffolk University · Sawyer Business School · Fall 2026 · Mon Dec 14 (exam-week slot) · Prof. Hasan Arslan**

Persist the pipeline, track it with MLflow, give it a face with Gradio, watch it drift with Evidently, know the rules — then present.

In [ ]:
import pandas as pd, numpy as np, json, joblib, sklearn, os
from sklearn.model_selection import train_test_split, TunedThresholdClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, confusion_matrix, roc_auc_score

URL = 'https://raw.githubusercontent.com/harslan/isom-835/master/public/data/telco_churn.csv'
df = pd.read_csv(URL)
COLS = ['tenure', 'MonthlyCharges', 'Contract', 'InternetService', 'PaymentMethod']
X, y = df[COLS], (df['Churn'] == 'Yes').astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=835)

## 1. Ship the pipeline, not the model
Preprocessing and estimator together, with the tuned threshold from Session 6. Scoring a new customer must build features exactly as training did.

In [ ]:
OFFER, SAVE = 50, 450
def profit(yt, yp): tn, fp, fn, tp = confusion_matrix(yt, yp).ravel(); return SAVE * tp - OFFER * (tp + fp)
prep = ColumnTransformer([('num', StandardScaler(), ['tenure', 'MonthlyCharges']), ('cat', OneHotEncoder(handle_unknown='ignore'), ['Contract', 'InternetService', 'PaymentMethod'])])
pipe = Pipeline([('prep', prep), ('clf', LogisticRegression(max_iter=2000))])
model = TunedThresholdClassifierCV(pipe, scoring=make_scorer(profit), cv=5).fit(X_tr, y_tr)
print(f'test AUC {roc_auc_score(y_te, model.predict_proba(X_te)[:, 1]):.3f}   tuned threshold {model.best_threshold_:.3f}   test profit ${profit(y_te, model.predict(X_te)):,}')

In [ ]:
joblib.dump(model, 'churn_pipeline_v1.joblib')
meta = {'sklearn': sklearn.__version__, 'threshold': float(model.best_threshold_), 'features': COLS, 'trained': '2026-12-14', 'test_auc': float(roc_auc_score(y_te, model.predict_proba(X_te)[:, 1]))}
json.dump(meta, open('churn_pipeline_v1.json', 'w'), indent=2)
print(os.path.getsize('churn_pipeline_v1.joblib'), 'bytes'); print(meta)

In [ ]:
# In a fresh process: load and score ONE customer from a dict — this is what the production service does
pipe2 = joblib.load('churn_pipeline_v1.joblib'); meta2 = json.load(open('churn_pipeline_v1.json'))
customer = {'tenure': 3, 'MonthlyCharges': 89.5, 'Contract': 'Month-to-month', 'InternetService': 'Fiber optic', 'PaymentMethod': 'Electronic check'}
p = float(pipe2.predict_proba(pd.DataFrame([customer]))[0, 1])
print(f'{p:.1%} churn risk →', 'SEND RETENTION OFFER' if p >= meta2['threshold'] else 'no action')

## 2. Track it with MLflow
The experiment log that replaces `final_v3_REAL.ipynb`: parameters, metrics, and the model artifact, every run.

In [ ]:
# OPTIONAL — pip install mlflow
try:
    import mlflow, mlflow.sklearn
    mlflow.set_experiment('isom835-churn')
    with mlflow.start_run(run_name='logistic-tuned-threshold'):
        mlflow.log_params({'model': 'LogisticRegression', 'features': ','.join(COLS), 'threshold': meta['threshold'], 'offer': OFFER, 'save': SAVE})
        mlflow.log_metrics({'test_auc': meta['test_auc'], 'test_profit': profit(y_te, model.predict(X_te))})
        mlflow.sklearn.log_model(model, name='model')
    print('logged. In Colab, run `!mlflow ui` (or view mlruns/) to compare runs')
except ImportError:
    print('mlflow not installed')

## 3. Give it a face with Gradio
A shareable web demo from inside Colab. A business owner will find the model's failure modes in thirty seconds.

In [ ]:
# OPTIONAL — pip install gradio   (launch(share=True) gives a public link from Colab)
try:
    import gradio as gr
    def score(tenure, monthly, contract, internet, payment):
        row = pd.DataFrame([{'tenure': tenure, 'MonthlyCharges': monthly, 'Contract': contract, 'InternetService': internet, 'PaymentMethod': payment}])
        p = float(pipe2.predict_proba(row)[0, 1])
        return f'{p:.1%} churn risk — ' + ('SEND RETENTION OFFER' if p >= meta2['threshold'] else 'no action')
    demo = gr.Interface(score, inputs=[gr.Slider(0, 72, 12, label='Tenure (months)'), gr.Slider(18, 120, 70, label='Monthly charges'),
                                       gr.Dropdown(['Month-to-month', 'One year', 'Two year'], value='Month-to-month', label='Contract'),
                                       gr.Dropdown(['DSL', 'Fiber optic', 'No'], value='Fiber optic', label='Internet'),
                                       gr.Dropdown(['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)'], value='Electronic check', label='Payment')],
                        outputs='text', title='Telco churn — ISOM 835')
    print('built. Uncomment to launch:  demo.launch(share=True)')
    # demo.launch(share=True)
except ImportError:
    print('gradio not installed')

## 4. Watch it drift with Evidently
Models rot. Compare this month's inputs to the training data; label delay means you watch inputs before you can watch accuracy. We simulate a shift: shorter tenures, more month-to-month contracts.

In [ ]:
shifted = X_te.copy(); rng = np.random.default_rng(835)
shifted['tenure'] = (shifted['tenure'] * 0.6).round(); shifted.loc[rng.random(len(shifted)) < 0.3, 'Contract'] = 'Month-to-month'
# Model-free drift check that always runs: population stability index per numeric column, share change per category
def psi(a, b, bins=10):
    edges = np.quantile(a, np.linspace(0, 1, bins + 1)); edges[0], edges[-1] = -np.inf, np.inf
    pa = np.histogram(a, edges)[0] / len(a) + 1e-6; pb = np.histogram(b, edges)[0] / len(b) + 1e-6
    return float(np.sum((pb - pa) * np.log(pb / pa)))
for c in ['tenure', 'MonthlyCharges']: print(f'PSI {c:15s} {psi(X_tr[c], shifted[c]):.3f}   ({"DRIFT" if psi(X_tr[c], shifted[c]) > 0.2 else "stable"})')
print('Month-to-month share: train', round((X_tr['Contract'] == 'Month-to-month').mean(), 3), '→ now', round((shifted['Contract'] == 'Month-to-month').mean(), 3))
print(f'predicted churn rate: train-like {model.predict_proba(X_te)[:, 1].mean():.3f} → shifted {model.predict_proba(shifted)[:, 1].mean():.3f}')

In [ ]:
# OPTIONAL — pip install evidently   (API for Evidently ≥ 0.6; produces an HTML report)
try:
    from evidently import Report
    from evidently.presets import DataDriftPreset
    report = Report([DataDriftPreset()]); snap = report.run(reference_data=X_tr, current_data=shifted)
    snap.save_html('drift_report.html'); print('drift_report.html written — open it in the Colab file browser')
except Exception as e:
    print('evidently not available or API differs →', type(e).__name__)

## 5. The rules of the road (one slide)
- **EU AI Act:** risk tiers; the Article 4 AI-literacy duty for providers and deployers has applied since Feb 2025 (enforceable Aug 2026); Article 50 transparency duties from Aug 2026; the 2026 Digital Omnibus deferred Annex III *high-risk* obligations (credit scoring, hiring, insurance) to **Dec 2, 2027**.
- **Colorado SB 26-189** (signed May 2026, effective **Jan 1, 2027**): disclosure, adverse-outcome explanation within 30 days, and human-review rights for automated decisions in consequential settings.
- If your model touches credit, hiring, insurance, or health: **explanation (Session 9), human review (conformal {both} sets), documentation (model card)** are the direction of travel everywhere.

## 6. Launch checklist
- [ ] Pipeline persisted with versions, threshold, feature list, training date
- [ ] Baseline beaten on a validation scheme that mirrors production
- [ ] Threshold or top-k tied to a cost matrix the business signed
- [ ] Calibration checked; conformal sets for the ambiguous cases
- [ ] Error rates sliced by group; model card written and owned
- [ ] Drift monitor scheduled; retraining trigger defined; outcome metric tracked when labels arrive

## 7. Your prediction story — presentation order
Eight minutes, two for questions: **question → data → model → honest evaluation → decision with a number → trust story.** The Signal Award vote follows the last talk.

Thank you for a wonderful semester. Find the signal.